# Retrieving a Reproducible Scientific Software Release with DSI and Zenodo

This notebook demonstrates how the **DSI Zenodo backend** can be used to retrieve
a versioned scientific software record from Zenodo, inspect its normalized
metadata and attached resources, and pass the selected software archive into a
downstream reproducibility workflow.

Unlike a domain-specific scientific analysis, this example focuses on a core
Zenodo use case: **preserving and citing an exact version of research software**.

## Why this user story?

The larger DOI collection used to identify potential Zenodo examples contains
more than 12,000 research records overall, including many Zenodo-associated
research outputs.

These include scientific software, datasets, simulations, notebooks, and model
results from areas such as:

- ocean modeling,
- plasma physics,
- climate science,
- quantum computing,
- geoscience,
- accelerator science,
- machine learning,
- astrophysics, and
- spectroscopy.

For this demonstration, **LANL WAVES** was selected because it provides a compact
software-oriented example.

Zenodo preserves the release as a citable research object with a persistent DOI
and an archived source-code resource.

## User Story

**As a computational researcher, I want to retrieve a specific archived version
of LANL scientific software from Zenodo through DSI so that I can identify the
exact software release, citation metadata, and source archive needed to support
a reproducible computational workflow.**

## Reproducibility Question

**Can DSI retrieve a specific citable WAVES software release from Zenodo and
identify the exact archived source artifact associated with that release?**

## Workflow

**Software DOI → DSI Zenodo backend → `datasets` metadata → `resources` metadata
→ source archive → download → checksum verification → archive inspection**

## What this notebook demonstrates

- retrieving a Zenodo software record by DOI;
- inspecting DSI-normalized record-level metadata;
- inspecting file-level resource metadata;
- identifying the archived source-code resource;
- downloading the resource using the URL exposed by DSI;
- verifying the downloaded archive against the checksum reported by Zenodo; and
- inspecting the archive contents to confirm that it contains a structured
  software release.

> **Important:** The Zenodo backend itself is read-only and does **not**
> automatically save attached research files to disk. It exposes the metadata
> and `download_url` required for a downstream workflow to retrieve the selected
> resource. The actual file download in this notebook is performed with
> `requests`.

## Environment and Dependencies

This example assumes that DSI has already been cloned and installed and that the
notebook-specific packages from `requirements.txt` are available.

The notebook uses:

- **DSI Zenodo backend** for repository access and metadata normalization;
- **pandas** for tabular inspection;
- **requests** for the downstream resource download;
- Python's built-in **hashlib** for checksum verification; and
- Python's built-in **zipfile** and **pathlib** modules for archive inspection
  and local file handling.

The Zenodo backend is the data-access layer.

Downloading, verification, and archive inspection remain downstream operations
so that repository access stays separate from use of the retrieved artifact.

In [1]:
# Standard-library utilities for local files, ZIP inspection,
# and checksum verification.

from pathlib import Path
from zipfile import ZipFile
import hashlib

# Third-party packages used by the downstream workflow.

import pandas as pd
import requests

# DSI backend used to retrieve and normalize the Zenodo record.

from dsi.backends.zenodo import Zenodo

print("Environment ready.")

Environment ready.


## 1. Select a Versioned WAVES Release

The selected Zenodo record is:

- **Title:** `lanl/waves: 0.6.21`
- **DOI:** `10.5281/zenodo.10016558`
- **Research object type:** software

Using a specific version DOI is important for reproducibility.

A software project may continue to change over time, but a version-specific
Zenodo DOI identifies an archived release that can be cited and retrieved
independently of later development.


In [2]:
waves_doi = "10.5281/zenodo.10016558"

print("Selected software DOI:", waves_doi)


Selected software DOI: 10.5281/zenodo.10016558


## 2. Retrieve the Software Record Through DSI

The Zenodo backend is a **read-only DSI Webserver backend**.

For this workflow, it uses the Zenodo Records API to retrieve the record
associated with the supplied DOI and normalize the returned repository metadata.

The backend exposes two stable tables:

- `datasets` — record-level metadata
- `resources` — file-level metadata

For a software release, the `datasets` table describes the release itself, while
the `resources` table identifies the files attached to that release.


In [3]:
zenodo = Zenodo(
    params={
        "doi": waves_doi
    },
    verify_ssl=False
)

print("Created Zenodo backend instance.")

/Users/rsankararaman/PDB_new/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'zenodo.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/rsankararaman/PDB_new/venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'zenodo.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Created Zenodo backend instance.


In [4]:
zenodo.list()

datasets: (1 rows, 23 cols)
resources: (1 rows, 14 cols)


In [5]:
datasets = zenodo.get_table(
    "datasets"
)

resources = zenodo.get_table(
    "resources"
)

print(
    "Datasets shape:",
    datasets.shape
)

print(
    "Resources shape:",
    resources.shape
)

Datasets shape: (1, 23)
Resources shape: (1, 14)


### How DSI Represents a Zenodo Record

The Zenodo backend separates each research object into two stable tables.

### `datasets`

The `datasets` table contains **record-level metadata**.

Examples include:

- DOI,
- title,
- version,
- creators,
- license,
- publication date,
- access rights, and
- number of attached resources.

### `resources`

The `resources` table contains **file-level metadata** for resources attached to
the Zenodo record.

Examples include:

- file name,
- file format,
- file size,
- checksum,
- MIME type, and
- download URL.

The two tables are connected through `dataset_id`.

This separation is useful because a single Zenodo record may contain one or many
files with different formats and purposes.


## 3. Inspect the Archived Software Metadata

A reproducible software citation requires more than the name of a software
project.

For a versioned software release, useful information includes:

- DOI,
- software version,
- publication date,
- creators,
- license,
- resource type,
- access rights, and
- number of archived resources.

DSI exposes these fields through the normalized `datasets` table.

In [6]:
software_metadata_columns = [
    column
    for column in [
        "dataset_id",
        "doi",
        "title",
        "version",
        "publication_date",
        "resource_type",
        "access_right",
        "license",
        "creators",
        "resource_count",
        "landing_page",
    ]
    if column in datasets.columns
]

software_metadata = datasets[
    software_metadata_columns
].copy()

software_metadata

,dataset_id,doi,title,version,publication_date,resource_type,access_right,license,creators,resource_count,landing_page
0,10016558,10.5281/zenodo.10016558,lanl/waves: 0.6.21,0.6.21,2023-10-17,"{'title': 'Software', 'type': 'software'}",open,{'id': 'cc-by-4.0'},"[{'name': 'Kyle Brindley', 'affiliation': 'Los...",1,https://zenodo.org/records/10016558


The returned record confirms that DSI retrieved the specific WAVES release
associated with the requested DOI.

The important point is that the backend is not returning only an opaque Zenodo
JSON response. It exposes commonly useful fields in a normalized tabular form
that can be inspected using standard DSI and pandas workflows.


## 4. Identify the Archived Source Resource

The record metadata tells us **which software release** was retrieved.

The `resources` table tells us **which files belong to that release**.

For this example, the key resource is the archived WAVES source-code ZIP file.

The resource table also exposes metadata that is useful for reproducibility,
including the archive size, checksum, and download URL.

In [7]:
resource_columns = [
    column
    for column in [
        "resource_id",
        "dataset_id",
        "name",
        "format",
        "size",
        "checksum",
        "download_url",
    ]
    if column in resources.columns
]

resource_view = resources[
    resource_columns
].copy()

resource_view

,resource_id,dataset_id,name,format,size,checksum,download_url
0,10016558:1,10016558,lanl/waves-0.6.21.zip,zip,407368,md5:e90f3f2f9fe1b7ea2d7eca4786e38563,https://zenodo.org/api/records/10016558/files/...


In [8]:
archive_resources = resources[
    resources["format"]
    .astype(str)
    .str.lower()
    .isin([
        "zip",
        "tar",
        "gz",
        "tar.gz"
    ])
].copy()

print(
    "Archived software resources found:",
    len(archive_resources)
)

archive_resources[
    [
        column
        for column in [
            "name",
            "format",
            "size",
            "checksum",
            "download_url",
        ]
        if column in archive_resources.columns
    ]
]

Archived software resources found: 1


,name,format,size,checksum,download_url
0,lanl/waves-0.6.21.zip,zip,407368,md5:e90f3f2f9fe1b7ea2d7eca4786e38563,https://zenodo.org/api/records/10016558/files/...


For this release, DSI identifies one archived software resource.

This is an important distinction in the workflow:

- the Zenodo record represents the **versioned research object**;
- the ZIP file represents the **actual archived software artifact**.

DSI connects these through the normalized `dataset_id` relationship.

## 5. Select and Download the Exact Archived Release

The source archive is selected from the DSI `resources` table rather than by
manually copying a file URL from the Zenodo website.

This preserves the relationship:

**software DOI → normalized record → archived resource**

The selection of the archive comes from DSI.

The actual file download in the next step is a **downstream notebook operation**.
The notebook uses the DSI-provided `download_url` with `requests` to save the
resource locally.

The Zenodo backend itself does not write the file to disk.

In [9]:
if archive_resources.empty:
    raise RuntimeError(
        "No software archive was found in the Zenodo resources table."
    )

selected_archive = archive_resources.iloc[0]

archive_name = selected_archive[
    "name"
]

archive_url = selected_archive[
    "download_url"
]

expected_checksum = selected_archive.get(
    "checksum"
)

print(
    "Selected archive:",
    archive_name
)

print(
    "Expected checksum:",
    expected_checksum
)

print(
    "Download URL:",
    archive_url
)

Selected archive: lanl/waves-0.6.21.zip
Expected checksum: md5:e90f3f2f9fe1b7ea2d7eca4786e38563
Download URL: https://zenodo.org/api/records/10016558/files/lanl/waves-0.6.21.zip/content


In [10]:
# Create a directory for the downloaded software release.

download_dir = Path(
    "downloaded_waves_release"
)

download_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Zenodo resource names may contain "/" characters.
# Replace them so that the resource can be stored as one local file.

safe_archive_name = archive_name.replace(
    "/",
    "_"
)

archive_path = (
    download_dir
    / safe_archive_name
)

# Download the resource using the URL exposed by the DSI resources table.

response = requests.get(
    archive_url,
    timeout=60
)

response.raise_for_status()

archive_path.write_bytes(
    response.content
)

print(
    "Zenodo resource name:",
    archive_name
)

print(
    "Local filename:",
    safe_archive_name
)

print(
    "Downloaded:",
    archive_path
)

print(
    "File size:",
    f"{archive_path.stat().st_size / 1024:.1f} KB"
)

Zenodo resource name: lanl/waves-0.6.21.zip
Local filename: lanl_waves-0.6.21.zip
Downloaded: downloaded_waves_release/lanl_waves-0.6.21.zip
File size: 397.8 KB


## 6. Verify the Downloaded Artifact

Zenodo provides a checksum for archived resources.

The checksum allows us to verify that the file downloaded in the notebook is the
same artifact described by the Zenodo resource metadata.

In this workflow:

- the **DOI** identifies the software release;
- the **resource metadata** identifies the associated archive; and
- the **checksum** verifies the integrity of the downloaded file.


In [11]:
def calculate_md5(path):
    md5 = hashlib.md5()

    with open(
        path,
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):
            md5.update(
                chunk
            )

    return md5.hexdigest()


actual_md5 = calculate_md5(
    archive_path
)

print(
    "Calculated MD5:",
    actual_md5
)

if isinstance(
    expected_checksum,
    str
):

    expected_md5 = (
        expected_checksum
        .replace(
            "md5:",
            ""
        )
        .strip()
    )

    print(
        "Expected MD5:  ",
        expected_md5
    )

    print(
        "Checksum match:",
        actual_md5 == expected_md5
    )

Calculated MD5: e90f3f2f9fe1b7ea2d7eca4786e38563
Expected MD5:   e90f3f2f9fe1b7ea2d7eca4786e38563
Checksum match: True


### Interpreting the Checksum Result

A `True` checksum match confirms that the locally downloaded archive is
byte-for-byte consistent with the MD5 checksum exposed by the Zenodo resource
metadata.

This provides a simple reproducibility check:

**persistent DOI → archived resource → verified local artifact**

The checksum does not evaluate the scientific correctness of the software. It
only confirms that the retrieved file matches the archived resource described by
Zenodo.

## 7. Inspect the Archived Software Release

The goal of this user story is not to install or execute WAVES.

Instead, the notebook performs a lightweight inspection of the downloaded ZIP
archive to confirm that it contains a structured software release.

This keeps the demonstration focused on:

**repository access → resource identification → reproducible retrieval**

rather than turning it into a WAVES installation or execution tutorial.

In [12]:
if archive_path.suffix.lower() != ".zip":

    print(
        "Archive inspection in this demo is implemented for ZIP files."
    )

else:

    with ZipFile(
        archive_path,
        "r"
    ) as archive:

        archived_files = archive.namelist()

    print(
        "Files/directories in archive:",
        len(archived_files)
    )

    print()
    print(
        "First 25 archive entries:"
    )

    for name in archived_files[:25]:
        print(
            "-",
            name
        )

Files/directories in archive: 315

First 25 archive entries:
- lanl-waves-1b9dee0/
- lanl-waves-1b9dee0/.gitattributes
- lanl-waves-1b9dee0/.github/
- lanl-waves-1b9dee0/.github/workflows/
- lanl-waves-1b9dee0/.github/workflows/pages.yml
- lanl-waves-1b9dee0/.github/workflows/release.yml
- lanl-waves-1b9dee0/.gitignore
- lanl-waves-1b9dee0/.gitlab-ci.yml
- lanl-waves-1b9dee0/CITATION.bib
- lanl-waves-1b9dee0/CITATION.cff
- lanl-waves-1b9dee0/LICENSE.txt
- lanl-waves-1b9dee0/MANIFEST.in
- lanl-waves-1b9dee0/README.rst
- lanl-waves-1b9dee0/SConscript
- lanl-waves-1b9dee0/SConstruct
- lanl-waves-1b9dee0/docs/
- lanl-waves-1b9dee0/docs/SConscript
- lanl-waves-1b9dee0/docs/_static/
- lanl-waves-1b9dee0/docs/_static/custom.css
- lanl-waves-1b9dee0/docs/_static/index.html
- lanl-waves-1b9dee0/docs/aea_environment_activation.txt
- lanl-waves-1b9dee0/docs/aea_release_discussion.txt
- lanl-waves-1b9dee0/docs/api.rst
- lanl-waves-1b9dee0/docs/build.txt
- lanl-waves-1b9dee0/docs/changelog.rst


The archive contains recognizable software-project content such as:

- source-code directories,
- documentation,
- citation files,
- licensing information,
- build/configuration files, and
- example or input files.

This confirms that the selected Zenodo resource is a preserved software source
release rather than an unrelated attachment.

In [13]:
if archive_path.suffix.lower() == ".zip":

    suffix_counts = {}

    for name in archived_files:

        path = Path(
            name
        )

        if name.endswith(
            "/"
        ):
            continue

        suffix = (
            path.suffix.lower()
            if path.suffix
            else "[no extension]"
        )

        suffix_counts[
            suffix
        ] = (
            suffix_counts.get(
                suffix,
                0
            )
            + 1
        )

    archive_summary = (
        pd.DataFrame(
            [
                {
                    "file_type": suffix,
                    "file_count": count,
                }
                for suffix, count
                in suffix_counts.items()
            ]
        )
        .sort_values(
            "file_count",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    archive_summary.head(
        15
    )

### What the Archive Inspection Shows

The archive contains a structured software release rather than a single opaque
binary file.

The presence of Python source files, documentation, citation files, licensing
information, build/configuration files, and other supporting resources shows
that the Zenodo artifact is a preserved software snapshot suitable for
downstream inspection or reproducibility workflows.

The notebook intentionally stops at archive inspection.

Installing, building, or executing WAVES would be a separate software-use
workflow and is outside the scope of this Zenodo backend demonstration.

## 8. Role of DSI in the Workflow

DSI serves as the **repository-access and organization layer** in this example.

### Without the DSI Zenodo Backend

A researcher would need to work directly with the Zenodo website or API to:

1. locate the correct software record;
2. confirm the intended software version;
3. interpret Zenodo-specific metadata;
4. identify the attached software archive;
5. obtain the file's download URL;
6. retain the checksum and other resource metadata; and
7. connect the downloaded file back to the cited release.

### With the DSI Zenodo Backend

The workflow becomes:

**DOI → DSI Zenodo → `datasets` / `resources` → selected archive → downstream verification**

In this demonstration:

- **`datasets`** identifies the versioned WAVES software record;
- **`resources`** identifies the exact archived source artifact;
- **DOI and version metadata** establish which software release was retrieved;
- **checksum and download URL** provide the information required to retrieve and
  verify the corresponding file; and
- standard Python tools perform the downstream download, checksum validation,
  and archive inspection.

The backend does not replace Zenodo, GitHub, package managers, or software build
tools.

Its role is to provide a consistent DSI representation of Zenodo research
objects so that repository-specific access remains separate from whatever
scientific or software workflow happens afterward.

## Conclusion

This user story demonstrates an end-to-end reproducibility workflow for a
versioned scientific software release stored in Zenodo.

A specific LANL WAVES release is retrieved by DOI through the DSI Zenodo
backend.

DSI exposes:

- normalized release metadata in `datasets`; and
- the attached source archive in `resources`.

The notebook then uses the DSI-provided resource metadata to download the
archive, verify its checksum, and inspect its contents.

The complete workflow can be summarized as:

**Identify → Retrieve metadata → Resolve resource → Download → Verify → Inspect**

The main value of the Zenodo backend is not to install or execute the software
itself.

Instead, it provides a consistent and reproducible bridge between a citable
Zenodo research object and the downstream tools that use its archived
resources.

In [14]:
zenodo.close()

print(
    "Zenodo backend closed."
)

Zenodo backend closed.
